This notebook computes metrics to be used later in graphs and LLMs

In [1]:
import pandas as pd
import numpy as np

# ================================
# 1. Load Data
# ================================

df = pd.read_csv('../data/analytics_base.csv')

# Convert dates
df['signup_date'] = pd.to_datetime(df['signup_date'], errors='coerce')
df['churn_date'] = pd.to_datetime(df['churn_date'], errors='coerce')

# Create time column (monthly based on signup)
df['month'] = df['signup_date'].dt.to_period('M').astype(str)

# ================================
# 2. Revenue Metrics
# ================================

monthly_revenue = df.groupby('month')['total_mrr'].sum().reset_index()
monthly_revenue.rename(columns={'total_mrr': 'monthly_mrr'}, inplace=True)

# Growth %
monthly_revenue['mrr_growth_pct'] = monthly_revenue['monthly_mrr'].pct_change() * 100

# Total revenue
total_mrr = df['total_mrr'].sum()
total_arr = df['total_arr'].sum()

# ================================
# 3. Churn Metrics
# ================================

# Total accounts
total_accounts = df['account_id'].nunique()

# Churned accounts
churned_accounts = df[df['churn_flag'] == True]['account_id'].nunique()

# Churn rate
churn_rate = churned_accounts / total_accounts if total_accounts > 0 else 0

# Monthly churn (based on churn_date)
churn_df = df.dropna(subset=['churn_date']).copy()
churn_df['churn_month'] = churn_df['churn_date'].dt.to_period('M').astype(str)

monthly_churn = churn_df.groupby('churn_month')['account_id'].count().reset_index()
monthly_churn.rename(columns={
    'churn_month': 'month',
    'account_id': 'monthly_churn'
}, inplace=True)

# ================================
# 4. Usage Metrics
# ================================

# Total usage
total_usage = df['usage_count_total'].sum()

# Avg usage per account
avg_usage_per_account = df['usage_count_total'].mean()

# Feature adoption
avg_features_used = df['feature_count'].mean()

# ================================
# 5. Support Metrics
# ================================

# Total tickets
total_tickets = df['ticket_count'].sum()

# Avg tickets per account
tickets_per_account = df['ticket_count'].mean()

# Avg resolution time
avg_resolution_time = df['avg_resolution_time'].mean()

# Escalation rate
total_escalations = df['escalated_tickets'].sum()
escalation_rate = total_escalations / total_tickets if total_tickets > 0 else 0

# ================================
# 6. Merge Metrics
# ================================

metrics_summary = pd.merge(monthly_revenue, monthly_churn, on='month', how='left')

# Fill missing churn values
metrics_summary['monthly_churn'] = metrics_summary['monthly_churn'].fillna(0)

# Add global KPIs
metrics_summary['total_mrr'] = total_mrr
metrics_summary['total_arr'] = total_arr
metrics_summary['churn_rate'] = churn_rate
metrics_summary['avg_usage_per_account'] = avg_usage_per_account
metrics_summary['avg_features_used'] = avg_features_used
metrics_summary['tickets_per_account'] = tickets_per_account
metrics_summary['avg_resolution_time'] = avg_resolution_time
metrics_summary['escalation_rate'] = escalation_rate

# ================================
# 7. Save Output
# ================================

metrics_summary.to_csv('../data/metrics_summary.csv', index=False)

print("✅ metrics_summary.csv created successfully!")

# ================================
# 8. Preview
# ================================

print(metrics_summary.head())

✅ metrics_summary.csv created successfully!
     month  monthly_mrr  mrr_growth_pct  monthly_churn  total_mrr  total_arr  \
0  2023-01       312351             NaN            0.0   11338747  136064964   
1  2023-02       330669        5.864556            0.0   11338747  136064964   
2  2023-03       440295       33.152790            2.0   11338747  136064964   
3  2023-04       236889      -46.197663            1.0   11338747  136064964   
4  2023-05       607673      156.522253            3.0   11338747  136064964   

   churn_rate  avg_usage_per_account  avg_features_used  tickets_per_account  \
0        0.22                 501.05             27.616                  4.0   
1        0.22                 501.05             27.616                  4.0   
2        0.22                 501.05             27.616                  4.0   
3        0.22                 501.05             27.616                  4.0   
4        0.22                 501.05             27.616                  4.